# Steps 2-4: Preprocessing, Model Setup, and Training

This combined notebook avoids repeating preprocessing. In Colab, keep the dataset zip in Google Drive, copy that one zip to `/content`, unzip locally, train from local disk, and save outputs/checkpoints back to Drive.


## Step 2: Preprocessing and Augmentation

In [ ]:
# install missing packages in colab if needed
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ["torch", "torchvision", "sklearn", "tqdm"] if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "scikit-learn", "tqdm"])

In [ ]:
import copy
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedShuffleSplit
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

In [ ]:
# mount drive, copy zip to local colab disk, then unzip locally
import shutil
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/proj3')
except Exception:
    DRIVE_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()

DRIVE_ZIP_PATH = DRIVE_PROJECT_ROOT / 'cifake.zip'
if DRIVE_ZIP_PATH is None:
    raise FileNotFoundError(
        f'Put the dataset zip at "MyDrive/proj3/cifake.zip"'
    )

PROJECT_ROOT = Path('/content/proj3_runtime')
LOCAL_ZIP_PATH = PROJECT_ROOT / DRIVE_ZIP_PATH.name
LOCAL_EXTRACT_ROOT = PROJECT_ROOT / 'data_unzipped'
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'output'
MODEL_DIR = OUTPUT_DIR / 'models'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# copy one large zip from drive instead of reading 120k files from drive
if not LOCAL_ZIP_PATH.exists() or LOCAL_ZIP_PATH.stat().st_size != DRIVE_ZIP_PATH.stat().st_size:
    print('copying zip to local colab disk...')
    shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
else:
    print('local zip already copied')

# unzip on local disk so training reads local files
if not any((root / 'train').is_dir() and (root / 'test').is_dir() for root in [LOCAL_EXTRACT_ROOT, *LOCAL_EXTRACT_ROOT.glob('*')]):
    print('unzipping locally...')
    LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['unzip', '-q', '-o', str(LOCAL_ZIP_PATH), '-d', str(LOCAL_EXTRACT_ROOT)])
else:
    print('local data already unzipped')

# find the folder that contains train/ and test/
candidate_data_roots = [LOCAL_EXTRACT_ROOT, *LOCAL_EXTRACT_ROOT.glob('*')]
DATA_ROOT = next(
    (root for root in candidate_data_roots if (root / 'train').is_dir() and (root / 'test').is_dir()),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find train/ and test/ inside the unzipped dataset.')

TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

print('drive project root:', DRIVE_PROJECT_ROOT)
print('local project root:', PROJECT_ROOT)
print('data root:', DATA_ROOT)
print('output dir:', OUTPUT_DIR)


In [ ]:
# check expected folder layout
required_dirs = [
    TRAIN_DIR / 'REAL',
    TRAIN_DIR / 'FAKE',
    TEST_DIR / 'REAL',
    TEST_DIR / 'FAKE',
]
missing_dirs = [str(folder) for folder in required_dirs if not folder.is_dir()]
if missing_dirs:
    raise FileNotFoundError(f'Missing expected folders: {missing_dirs}')

In [ ]:
# choose device
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('gpu:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('gpu not found, using cpu')

In [ ]:
# preprocessing and training config
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
MAX_EPOCHS = 6
HEAD_ONLY_EPOCHS = 2
PATIENCE = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODELS_TO_TRAIN = ['resnet50', 'densenet121']

# use small numbers for a smoke test, or None for the full dataset
LIMIT_TRAIN_SAMPLES = None
LIMIT_VAL_SAMPLES = None

In [ ]:
# define image transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
# build datasets and split validation from training set
full_train_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
full_train_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

labels = np.array([label for _, label in full_train_aug.samples])
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(np.zeros(len(labels)), labels))

if LIMIT_TRAIN_SAMPLES:
    train_idx = train_idx[:LIMIT_TRAIN_SAMPLES]
if LIMIT_VAL_SAMPLES:
    val_idx = val_idx[:LIMIT_VAL_SAMPLES]

train_dataset = Subset(full_train_aug, train_idx)
val_dataset = Subset(full_train_eval, val_idx)
class_names = full_train_aug.classes

print('classes:', class_names)
print('class to index:', full_train_aug.class_to_idx)
print('train images:', len(train_dataset))
print('validation images:', len(val_dataset))
print('test images:', len(test_dataset))

In [ ]:
# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('train batches:', len(train_loader))
print('validation batches:', len(val_loader))
print('test batches:', len(test_loader))

In [ ]:
# check class counts
def class_counts(dataset):
    if isinstance(dataset, Subset):
        targets = [dataset.dataset.samples[i][1] for i in dataset.indices]
        classes = dataset.dataset.classes
    else:
        targets = dataset.targets
        classes = dataset.classes
    return {classes[i]: targets.count(i) for i in sorted(set(targets))}

print('train class counts:', class_counts(train_dataset))
print('validation class counts:', class_counts(val_dataset))
print('test class counts:', class_counts(test_dataset))

In [ ]:
# view one processed batch
images, labels = next(iter(train_loader))
print('image batch shape:', images.shape)
print('label batch shape:', labels.shape)

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.ravel(), images[:8], labels[:8]):
    image = (image.cpu() * std + mean).clamp(0, 1)
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(class_names[int(label)])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# save preprocessing settings
preprocess_config = {
    'image_size': IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'batch_size': BATCH_SIZE,
    'train_size': len(train_dataset),
    'val_size': len(val_dataset),
    'test_size': len(test_dataset),
    'classes': class_names,
    'class_to_idx': full_train_aug.class_to_idx,
    'seed': SEED,
}

config_path = OUTPUT_DIR / 'preprocess_config.json'
with open(config_path, 'w') as f:
    json.dump(preprocess_config, f, indent=2)
print('saved:', config_path)

## Step 3: Model Setup

In [ ]:
# build model and replace classifier
def build_model(model_name, num_classes=2):
    if model_name == 'resnet50':
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        classifier_params = model.fc.parameters()
    elif model_name == 'densenet121':
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
        classifier_params = model.classifier.parameters()
    else:
        raise ValueError(f'Unknown model: {model_name}')

    return model, classifier_params


def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

## Step 4: Training and Validation

In [ ]:
# train or validate for one epoch
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_correct += (preds == labels).sum().item()
        total_items += images.size(0)

    return total_loss / total_items, total_correct / total_items

In [ ]:
# train one model with early stopping
def train_model(model_name):
    model, classifier_params = build_model(model_name, num_classes=len(class_names))
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler(enabled=torch.cuda.is_available())

    freeze_backbone(model)
    for param in classifier_params:
        param.requires_grad = True

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_loss = float('inf')
    best_state = None
    bad_epochs = 0
    history = []
    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == HEAD_ONLY_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE / 10, weight_decay=WEIGHT_DECAY)

        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)

        row = {
            'model': model_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'elapsed_min': (time.time() - start_time) / 60,
        }
        history.append(row)
        print(row)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
            checkpoint_path = MODEL_DIR / f'{model_name}_best.pt'
            torch.save({
                'model_name': model_name,
                'model_state_dict': best_state,
                'class_names': class_names,
                'image_size': IMG_SIZE,
                'imagenet_mean': IMAGENET_MEAN,
                'imagenet_std': IMAGENET_STD,
                'val_loss': best_val_loss,
                'val_acc': val_acc,
            }, checkpoint_path)
            print('saved best checkpoint:', checkpoint_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print('early stopping')
                break

    history_df = pd.DataFrame(history)
    history_path = OUTPUT_DIR / f'{model_name}_training_history.csv'
    history_df.to_csv(history_path, index=False)
    print('saved history:', history_path)

    return history_df

In [ ]:
# run training
all_histories = []
for model_name in MODELS_TO_TRAIN:
    print()
    print('training', model_name)
    history_df = train_model(model_name)
    all_histories.append(history_df)

combined_history = pd.concat(all_histories, ignore_index=True)
combined_history_path = OUTPUT_DIR / 'training_history_all.csv'
combined_history.to_csv(combined_history_path, index=False)
print('saved combined history:', combined_history_path)
combined_history

In [ ]:
# plot validation accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name, group in combined_history.groupby('model'):
    axes[0].plot(group['epoch'], group['val_acc'], marker='o', label=model_name)
    axes[1].plot(group['epoch'], group['val_loss'], marker='o', label=model_name)

axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plot_path = OUTPUT_DIR / 'training_curves.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('saved plot:', plot_path)

In [ ]:
# list saved outputs
for output_path in sorted(OUTPUT_DIR.glob('*')):
    print(output_path)